# 📘 Text Summarization with Transformers — A Complete NLP & GenAI Learning Notebook

> Annotated by an NLP/GenAI instructor pass. Every original code cell is preserved exactly as it ran; markdown cells have been added throughout to explain **what** each step does, **why** it's needed, and **which concept** it teaches. Treat this notebook as a reusable reference for future NLP/GenAI projects.

## What are we building?

We are going to take a **pretrained Transformer model (Pegasus)** and **fine-tune** it to summarize **chat/dialogue conversations** (the SAMSum dataset — think WhatsApp/Messenger-style chats) into short, human-readable summaries.

```
Raw dialogue  --->  [Fine-tuned Pegasus model]  --->  1-3 sentence summary
```

## Two flavors of summarization (concept #1)

| Type | How it works | Example |
|---|---|---|
| **Extractive** | Picks and stitches together existing sentences/phrases from the source text. Never invents new words. | Classic algorithms like TextRank. |
| **Abstractive** | *Generates* new sentences that paraphrase the meaning, like a human would. Requires a language-generation model. | What we're doing here, with Pegasus. |

Abstractive summarization is a **sequence-to-sequence (seq2seq)** problem: input = long text, output = short text, both variable length. This is the same family of problem as machine translation, which is why the tools below (encoder-decoder Transformers) are shared across both tasks.

## Roadmap of this notebook

1. **Environment setup** — GPU check, installing libraries
2. **Quick demo** — using an off-the-shelf pretrained summarizer (no training)
3. **Fine-tuning** — adapting Pegasus to our specific dialogue-summarization dataset
4. **Evaluation** — scoring summary quality with the ROUGE metric
5. **Save & deploy** — persisting the model and running inference with it

## Prerequisites / mental model

You don't need to know Transformer internals in depth to use this notebook, but a few terms will recur constantly:
- **Tokenizer** — converts text ↔ numbers (token IDs) that a neural network can consume.
- **Token** — a sub-word unit (not always a whole word). E.g. "summarization" might become `["_summar", "ization"]`.
- **Encoder-decoder (seq2seq) Transformer** — one network (encoder) reads and compresses the input; another (decoder) generates the output token-by-token, attending back to the encoder's representation.
- **Fine-tuning** — continuing to train an already-pretrained model on a smaller, task-specific dataset so it specializes without learning language "from scratch."
- **GenAI connection** — this whole notebook is a *smaller-scale, specialized* cousin of what powers ChatGPT/Claude-style models: a Transformer trained to generate text conditioned on an input. The difference is scale (millions vs. billions of parameters) and generality (one task vs. many, via instructions/prompting).

Let's start.

---
## 1. Environment & Hardware Check

Transformer models are large stacks of matrix multiplications. A GPU can execute thousands of these multiplications in parallel (vs. a CPU's handful of cores), which is the difference between a training run taking **minutes** vs. **hours/days**. The `nvidia-smi` command below reports which GPU is attached to this runtime, its memory (VRAM), and current utilization — always check this first, since "out of memory" errors are one of the most common issues in deep learning.

In [ ]:
!nvidia-smi

Sat Oct 12 03:07:43 2024
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L4                      Off | 00000000:00:03.0 Off |                    0 |
| N/A   45C    P0              28W /  72W |  13013MiB / 23034MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+---------

## 2. Installing the Libraries

```
pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q
```

Each package plays a distinct role in the pipeline — worth knowing what each one is for, since you'll reuse this exact stack in almost every NLP fine-tuning project:

| Package | Role |
|---|---|
| **transformers** (Hugging Face) | Provides pretrained model architectures (Pegasus, BERT, T5, GPT, …), tokenizers, and training utilities (`Trainer`, `TrainingArguments`, `pipeline`). This is the core library of the whole notebook. |
| **sentencepiece** (`transformers[sentencepiece]`) | A tokenization *algorithm/library*. Many models (Pegasus, T5, XLNet) use SentencePiece sub-word tokenizers instead of the more common BPE/WordPiece — this extra installs the backend needed to load those tokenizers. |
| **datasets** (Hugging Face) | Efficient loading, caching, and processing of large datasets (works well with Pandas-like `.map()` operations, but backed by Apache Arrow for speed and low memory use). |
| **sacrebleu** | Implements the BLEU metric (standard for machine translation). Installed here because it's a common companion metric, even though this notebook focuses on ROUGE. |
| **rouge_score** | The actual scoring library behind the ROUGE metric (used later in Evaluation) — measures n-gram overlap between generated and reference summaries. |
| **py7zr** | Lets `datasets` unpack `.7z`-compressed dataset archives (some HF datasets ship compressed this way). |

`-q` just means "quiet" install output.

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

### Purpose of `accelerate`

1. **Ease of Multi-Device Training**: Whether you're using multiple GPUs or TPUs, `accelerate` makes it easier to scale your model across devices with minimal code changes.
2. **Mixed Precision**: It allows models to be trained using mixed precision, which can speed up training and reduce memory usage.
3. **Zero Redundancy Optimizer (ZeRO)**: Helps manage large models by efficiently splitting the model across multiple devices.
4. **Offload to CPU/SSD**: Useful for large models that may not fit entirely into GPU memory, by allowing parts of the model or optimizer to be offloaded to CPU or even SSD.

`accelerate` is the library the Hugging Face `Trainer` uses under the hood to actually dispatch tensors to the right device(s) — we install/upgrade it explicitly here because `Trainer` depends on a reasonably recent version.

**Why uninstall and reinstall `transformers`/`accelerate` right after installing `accelerate`?**

This is a common "notebook hygiene" pattern in Colab-style environments: Colab often ships a slightly different (older or newer) pre-installed version of `transformers` than the one that plays nicely with the `accelerate` version you just installed. Forcing a clean uninstall + reinstall guarantees both libraries are the *latest compatible pair*, avoiding cryptic version-mismatch errors later during training (a very common source of bugs when following any HF tutorial — if the `Trainer` throws a strange `TypeError` about unexpected keyword arguments, mismatched `transformers`/`accelerate` versions are the first thing to suspect).

⚠️ **Practical tip:** after any cell that upgrades/reinstalls core libraries in Colab/Jupyter, you often need to **restart the runtime/kernel** before the new versions are actually loaded into memory, since Python caches already-imported modules.

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

  Using cached accelerate-1.0.1-py3-none-any.whl.metadata (19 kB)
...
Found existing installation: accelerate 1.0.1
Uninstalling accelerate-1.0.1:
  Successfully uninstalled accelerate-1.0.1
  Using cached transformers-4.45.2-py3-none-any.whl.metadata (44 kB)
  Using cached accelerate-1.0.1-py3-none-any.whl.metadata (19 kB)
...


## 3. Core Imports — what each one is for

```python
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
```

- **`pipeline`** — Hugging Face's highest-level convenience API. Give it a task name (`"summarization"`, `"sentiment-analysis"`, etc.) plus a model, and it handles tokenization → model inference → decoding for you in one call. Great for quick demos and inference; not used for training.
- **`set_seed`** — fixes random seeds (Python, NumPy, PyTorch) so results are reproducible.
- **`load_dataset` / `load_from_disk`** — `load_dataset` pulls a dataset from the Hugging Face Hub (or a script); `load_from_disk` reloads a dataset previously saved locally via `.save_to_disk()`. We use the latter since the SAMSum data is downloaded as a pre-built Arrow dataset folder.
- **`AutoModelForSeq2SeqLM`** — an "auto class" that automatically picks the right *encoder-decoder* model architecture class based on the checkpoint name you give it (here, Pegasus). Using `Auto*` classes instead of importing `PegasusForConditionalGeneration` directly means the same code would work if you swapped in T5, BART, or another seq2seq model.
- **`AutoTokenizer`** — same idea, but for loading the matching tokenizer.
- **`nltk` / `sent_tokenize` / `nltk.download("punkt")`** — NLTK's **punkt** is a pretrained sentence-boundary detector (it knows that "Mr. Smith went to Washington." has one sentence, not two, despite the period after "Mr"). It's used for splitting text into sentences — useful for post-processing generated summaries or preparing extractive baselines.
- **`tqdm`** — a progress bar for loops, useful when we batch-generate summaries over the whole test set later.
- **`torch`** — PyTorch, the deep-learning framework `transformers` is built on here (it also supports TensorFlow/JAX backends, but PyTorch is what's running under the hood in this notebook).

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
#from datasets import load_dataset, load_metric

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## 4. Basic Functionality of a Pretrained Hugging Face Summarization Model

Before we fine-tune anything, let's understand the model architecture we're working with, since everything downstream builds on this.

### 4.1 The Transformer, encoder-decoder edition

A **Transformer** is a neural network architecture built almost entirely out of a mechanism called **self-attention**, which lets every token in a sequence directly "look at" every other token (instead of processing left-to-right like older RNN/LSTM models). This makes it both more parallelizable (fast to train on GPUs) and better at capturing long-range dependencies in text.

For summarization we use the **encoder-decoder** (seq2seq) variant:

```
Input dialogue ──► [ENCODER] ──► contextual representation ──► [DECODER] ──► Summary (generated token by token)
                                                                     ▲
                                                        also attends back to encoder output
                                                        ("cross-attention")
```

- The **encoder** reads the *entire* input at once and produces a contextual numeric representation of it (one vector per input token).
- The **decoder** generates the output **autoregressively** — one token at a time, where each new token is predicted based on (a) all previously generated tokens and (b) the encoder's representation of the source text via *cross-attention*.

### 4.2 What is Pegasus?

**Pegasus** ("Pre-training with Extracted Gap-sentences for Abstractive Summarization") is a Transformer specifically **pretrained for summarization** (Google Research, 2020). Its pretraining trick — Gap Sentence Generation (GSG) — masks out whole *important sentences* from a document and trains the model to generate exactly those missing sentences using the rest of the document as context. This is much closer to the actual summarization task than generic pretraining objectives (like masked-word prediction), which is why Pegasus tends to need relatively little fine-tuning data to become good at a specific summarization dataset.

Different **checkpoints** of Pegasus are fine-tuned on different datasets and therefore produce different *summary styles*:
- **`google/pegasus-xsum`** (used below) — fine-tuned on the XSum news dataset, which pairs articles with a **single, very short, one-sentence** summary. Expect terse, headline-like output.
- **`google/pegasus-cnn_dailymail`** (used later, for our own fine-tuning) — fine-tuned on CNN/DailyMail, which uses **multi-sentence, bullet-point-like** summaries. This style is a better starting point for the SAMSum task, where summaries are typically 1-3 sentences of narrative text.

### 4.3 What the demo cell below does

1. Load the model weights + config (`PegasusForConditionalGeneration.from_pretrained(...)`) and the matching tokenizer.
2. **Tokenize** the raw string into `input_ids` (integer token IDs) — this is the only format a neural network can consume.
3. Call `model.generate(...)` — this runs the encoder once, then runs the decoder in a loop, producing one token at a time until it emits an end-of-sequence token (or hits `max_length`). By default this uses **greedy/beam search decoding** (more on decoding strategies later, in the Inference section).
4. `tokenizer.batch_decode(...)` converts the generated token IDs back into a human-readable string, stripping special tokens (like end-of-sequence markers).

> Don't worry about the *"weights were not initialized... You should probably TRAIN this model"* warning you'll see in the output below — it refers to the positional-embedding weights for this specific checkpoint/version combo, not the core summarization weights, and the demo output is still coherent. Always read such warnings, but don't panic on sight — verify against the actual output quality, as we do here.

In [ ]:
from transformers import AutoTokenizer, PegasusForConditionalGeneration

model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-xsum")
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")

ARTICLE_TO_SUMMARIZE = (
    "PG&E stated it scheduled the blackouts in response to forecasts for high winds "
    "amid dry conditions. The aim is to reduce the risk of wildfires. Nearly 800 thousand customers were "
    "scheduled to be affected by the shutoffs which were expected to last through at least midday tomorrow."
)
inputs = tokenizer(ARTICLE_TO_SUMMARIZE, max_length=1024, return_tensors="pt")

# Generate Summary
summary_ids = model.generate(inputs["input_ids"])
tokenizer.batch_decode(summary_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


"California's largest electricity provider has turned off power to hundreds of thousands of customers."

**Reading the output:** the 3-sentence news snippet about PG&E blackouts got compressed into one clean sentence: *"California's largest electricity provider has turned off power to hundreds of thousands of customers."* Notice it's **abstractive** — that exact sentence doesn't appear verbatim in the source; the model paraphrased and combined facts (PG&E → "California's largest electricity provider", 800k customers → "hundreds of thousands"). That's the hallmark of abstractive summarization vs. simple sentence extraction.

Next we move to setting up the GPU device, then to the main event: fine-tuning this architecture on our own dataset.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

**Device placement.** PyTorch tensors and models live in a specific place in memory — either regular RAM (`"cpu"`) or GPU VRAM (`"cuda"`). Every tensor involved in a computation must be on the *same* device, or PyTorch raises an error. `torch.cuda.is_available()` checks whether a CUDA-capable GPU is visible to this process; we store the result in `device` so every later `.to(device)` call consistently sends models/tensors to the fastest available hardware. Since `nvidia-smi` showed an NVIDIA L4 GPU earlier, we expect `device == "cuda"` here.

## 5. Fine-Tuning — Adapting Pegasus to Our Task

### 5.1 What is fine-tuning, and why not train from scratch?

Training a Transformer from random initialization requires **enormous** amounts of text and compute (Pegasus itself was pretrained on hundreds of gigabytes of web/news text using industrial-scale hardware). **Fine-tuning** instead starts from those already-learned weights and continues training on a much smaller, task-specific dataset (here: ~14.7k SAMSum dialogue/summary pairs). This is the core idea of **transfer learning**: general language understanding transfers across tasks, so we only need to teach the model the *specifics* of our target task (dialogue → summary), not language itself.

### 5.2 Why start from `google/pegasus-cnn_dailymail` rather than `pegasus-xsum` or the base `pegasus-large`?

- `pegasus-large` (no downstream fine-tuning) only has the generic gap-sentence pretraining objective — it can technically generate text but isn't yet specialized for "produce a concise summary."
- `pegasus-xsum` is tuned for **single-sentence, extreme** summaries — a style mismatch for SAMSum, whose summaries are usually short narrative paragraphs (1-3 sentences describing what happened in the chat).
- `pegasus-cnn_dailymail` is tuned for **multi-sentence, narrative-style** summaries of news articles — structurally the closest existing skill to what we want, so fine-tuning from here should converge faster and need less data than starting further away.

This "pick the closest existing checkpoint" strategy is a good default heuristic whenever you fine-tune for a new but related task.

### 5.3 A note on scale — full fine-tuning vs. parameter-efficient fine-tuning (PEFT)

What we do below is **full fine-tuning**: every parameter in the ~570M-parameter Pegasus model is updated. This works well for models of this size, but for today's much larger LLMs (billions of parameters), practitioners often use **parameter-efficient fine-tuning** techniques like **LoRA** (Low-Rank Adaptation), which freeze the original weights and train small additional "adapter" matrices instead — dramatically cutting GPU memory and storage needs while achieving similar quality. Worth knowing the name even if we don't use it here; it's the standard approach for fine-tuning large GenAI models today.

In [ ]:
model = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model)  #load a tokenizer

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model).to(device)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**What's happening in this cell:**

```python
model = "google/pegasus-cnn_dailymail"                                    # just the checkpoint name (a string)
tokenizer = AutoTokenizer.from_pretrained(model)                          # load matching vocab + tokenization rules
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model).to(device)   # load weights, move to GPU
```

Note the two different variables: `model` (a plain string identifier used to *locate* the checkpoint on the Hugging Face Hub) vs. `model_pegasus` (the actual loaded `nn.Module` object with weights in memory). Keeping these separate is intentional — `model` (the string) is never used for computation again, while `model_pegasus` (the real object) is what gets passed to the data collator, the `Trainer`, and eventually saved to disk.

`.to(device)` moves every one of the model's weight tensors onto the GPU, so all subsequent forward/backward passes run there.

### 5.4 The SAMSum Dataset

**SAMSum** is a dataset of ~16,000 English messenger-style conversations (with linguists asked to write them, to mimic real chat style — slang, emojis, typos included), each paired with a human-written abstractive summary. It's the standard benchmark for **dialogue summarization** (as opposed to news-article summarization, which is what CNN/DailyMail and XSum cover). This is exactly why we need to *fine-tune* — the base `pegasus-cnn_dailymail` checkpoint has never seen chat-style text, only news prose.

The cell below downloads a zipped bundle containing:
- CSV versions of the splits (`samsum-train.csv`, `samsum-test.csv`, `samsum-validation.csv`)
- A pre-built Hugging Face `datasets` folder (`samsum_dataset/`) in Apache Arrow format — faster to load and what we'll actually use via `load_from_disk`.

In [ ]:
#dowload & unzip data

!wget https://github.com/krishnaik06/datasets/raw/refs/heads/main/summarizer-data.zip
!unzip summarizer-data.zip

--2024-10-12 03:18:44--  https://github.com/krishnaik06/datasets/raw/refs/heads/main/summarizer-data.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/krishnaik06/datasets/refs/heads/main/summarizer-data.zip [following]
--2024-10-12 03:18:45--  https://raw.githubusercontent.com/krishnaik06/datasets/refs/heads/main/summarizer-data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7903594 (7.5M) [application/zip]
Saving to: 'summarizer-data.zip'

summarizer-data.zip 100%[===================>]   7.54M  --.-KB/s    in 0.02s

2024-10-12 03:18:45 (341 MB/s) - 'summarizer-data.zip' saved [7903594/7903594]

Archive:  summarizer-data.

In [ ]:
dataset_samsum = load_from_disk('samsum_dataset')
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

**`DatasetDict` structure.** A Hugging Face `DatasetDict` is just a dictionary of `Dataset` objects keyed by split name (`train`/`test`/`validation`). Each `Dataset` behaves like a table: rows are examples, columns are features. Here every row has three columns: `id` (unique identifier), `dialogue` (the raw chat, with speaker names and turns), and `summary` (the human-written reference). Splits:

- **train** (14,732 examples) — used to update model weights.
- **validation** (818 examples) — used *during* training to monitor generalization (never used to update weights directly).
- **test** (819 examples) — held out entirely, used only for final evaluation after training is done.

Keeping these three separate is one of the most important disciplines in ML — watch for a callout later showing where this notebook actually breaks that discipline (as a speed shortcut) and what the correct practice looks like.

In [ ]:
split_lengths = [len(dataset_samsum[split])for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum["test"][2]["dialogue"])

print("\nSummary:")

print(dataset_samsum["test"][2]["summary"])

Split lengths: [14732, 819, 818]
Features: ['id', 'dialogue', 'summary']

Dialogue:
Lenny: Babe, can you help me with something?
Bob: Sure, what's up?
Lenny: Which one should I pick?
Bob: Send me photos
Lenny:  <file_photo>
Lenny:  <file_photo>
Lenny:  <file_photo>
Bob: I like the first ones best
Lenny: But I already have purple trousers. Does it make sense to have two pairs?
Bob: I have four black pairs :D :D
Lenny: yeah, but shouldn't I pick a different color?
Bob: what matters is what you'll give you the most outfit options
Lenny: So I guess I'll buy the first or the third pair then
Bob: Pick the best quality then
Lenny: ur right, thx
Bob: no prob :)

Summary:
Lenny can't decide which trousers to buy. Bob advised Lenny on that topic. Lenny goes with Bob's advice to pick the trousers that are of best quality.


Looking at this real example is worth pausing on: the dialogue is informal (short turns, `<file_photo>` placeholders for shared images, no punctuation discipline), while the summary is a clean, grammatical, third-person narrative. The model has to learn this *register shift* — not just "make it shorter," but "rewrite chat-speak as a coherent narrative description." This is a harder transformation than news summarization, where the input is already well-formed prose.

## 6. Preparing Data for a Seq2Seq Model

### 6.1 From raw text to model inputs

Neural networks only understand numbers, so every raw text field needs to become numeric **token IDs**. But a seq2seq training example needs *three* numeric arrays, not just one:

```
raw example:
{
    'dialogue': "Hi! How are you?",
    'summary': "The speaker is asking how the other person is."
}

        │  tokenize source and target separately
        ▼

model-ready example:
{
    'input_ids':      [123, 456, 789, ...],  # tokenized DIALOGUE — fed to the ENCODER
    'attention_mask': [1, 1, 1, ...],        # marks real tokens (1) vs. padding (0) for the encoder
    'labels':         [321, 654, 987, ...]   # tokenized SUMMARY — the target the DECODER must learn to produce
}
```

### 6.2 Why `labels` and not another `input_ids`?

Hugging Face seq2seq models expect the target sequence in a field literally named `labels`. Internally, the model:
1. Shifts `labels` right by one position to create `decoder_input_ids` (this is **teacher forcing** — during training, the decoder is fed the *correct* previous token rather than its own possibly-wrong prediction, which stabilizes and speeds up learning).
2. Computes the **cross-entropy loss** between its predicted next-token distribution at each position and the actual next token in `labels`.
3. Backpropagates that loss to update every weight in the encoder + decoder.

This "predict the next token, compare to ground truth, backprop" loop, repeated across the whole training set, *is* fine-tuning. Everything else in this notebook is plumbing around that core loop.

### 6.3 The tokenization function

```python
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(
        example_batch["dialogue"], max_length=1024, truncation=True
    )
    target_encodings = tokenizer(
        text_target=example_batch["summary"], max_length=128, truncation=True
    )
    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"],
    }
```

- We call the tokenizer **twice** — once on `dialogue` (the source) and once on `summary` (the target) — because each needs different truncation lengths. The modern, recommended way to tokenize the target sequence is the `text_target=` keyword argument (older tutorials use a deprecated `as_target_tokenizer()` context manager instead — you'll see a `UserWarning` about this below when the deprecated internal path still gets touched by some library internals).
- **`max_length=1024`** for dialogues: generous headroom, since conversations can run long; anything beyond this is **truncated** (cut off) — a real limitation to be aware of, since truncated context means the model never sees the end of very long conversations.
- **`max_length=128`** for summaries: SAMSum summaries are always short (1-3 sentences), so 128 tokens is comfortably more than needed while still capping worst-case runaway generations during training.
- `truncation=True` enables the max-length cutoff instead of erroring out on longer sequences.
- The function takes a *batch* of examples and returns a batch of results — required for the efficient `.map(..., batched=True)` call next, which processes many rows per function call instead of one Python-level call per row (much faster, especially since the tokenizer's core loop is implemented in Rust).

In [ ]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(
        example_batch["dialogue"], max_length=1024, truncation=True
    )

    # Modern Hugging Face approach: pass text_target directly
    target_encodings = tokenizer(
        text_target=example_batch["summary"], max_length=128, truncation=True
    )

    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"],
    }

In [ ]:
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched = True)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:4109: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
dataset_samsum_pt['test']

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 819
})

After `.map()`, every split now has three *new* columns (`input_ids`, `attention_mask`, `labels`) alongside the original `id`/`dialogue`/`summary` — the raw text columns are kept too, which is handy for inspection and for the evaluation step later where we need the raw strings again.

### 6.4 Dynamic padding with `DataCollatorForSeq2Seq`

Different examples produce different numbers of tokens. To stack multiple examples into one GPU batch (a rectangular tensor), all sequences in that batch must be the **same length** — so shorter ones get padded with a special padding token.

`DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)`, used just below, builds these padded batches **on the fly, per batch** ("dynamic padding") rather than padding the *entire dataset* to one global max length up front. This matters for efficiency: a batch of short dialogues stays short (less wasted compute on padding tokens), while only a batch that happens to contain a long dialogue pays the cost of extra padding.

Two extra things this collator handles automatically that are easy to forget if you build a collator by hand:
- **Encoder-side padding** (`input_ids`/`attention_mask`) uses the tokenizer's normal pad token.
- **Label-side padding** uses `-100` instead of the pad token ID — `-100` is a special value PyTorch's cross-entropy loss is configured to *ignore*. Without this, the model would be trained to predict "padding" as if it were a real target token, corrupting the loss signal.
- Because we pass `model=model_pegasus`, the collator also knows how to correctly construct `decoder_input_ids` (the right-shifted labels) for this specific architecture.

In [ ]:
# Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

## 7. Training with the `Trainer` API

⚠️ **Bug fix note:** the original notebook uses `TrainingArguments` and `Trainer` below without ever importing them anywhere. It only ran previously because some other cell in that Colab session happened to import them into memory earlier in the session. On a fresh run this would raise `NameError: name 'TrainingArguments' is not defined`. The next cell adds the missing import explicitly — always import everything you use, even if a stale kernel state lets you get away with skipping it once.

### 7.1 What is the `Trainer`?

`Trainer` is Hugging Face's high-level training loop abstraction. Instead of you hand-writing the standard PyTorch loop —

```python
for batch in dataloader:
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
```

— `Trainer` wraps all of that (plus gradient accumulation, mixed precision, periodic evaluation, checkpointing, logging, and multi-GPU/distributed handling via `accelerate` under the hood) behind a single `.train()` call. You configure its behavior via a companion `TrainingArguments` object, explained hyperparameter-by-hyperparameter below.

In [ ]:
from transformers import Trainer, TrainingArguments  # <-- added: required, was missing in the original notebook

### 7.2 `TrainingArguments`, hyperparameter by hyperparameter

```python
trainer_args = TrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16
)
```

| Argument | Meaning |
|---|---|
| `output_dir` | Where checkpoints, logs, and configs get written during training. |
| `num_train_epochs=1` | One full pass over the training data. In a real project you'd typically use 3–10 epochs for a dataset this size — `1` here is a deliberately fast/cheap demo setting. |
| `warmup_steps=500` | **Learning-rate warmup**: the learning rate ramps up linearly from 0 to its target value over the first 500 steps, instead of starting at full strength immediately. This avoids large, destabilizing weight updates early in training, when the optimizer's gradient statistics are still noisy. |
| `per_device_train_batch_size=1` / `per_device_eval_batch_size=1` | Number of examples processed at once, per GPU, before a weight update. `1` is very small — chosen here to fit comfortably in limited GPU memory given how long Pegasus's dialogue inputs can be (up to 1024 tokens each). |
| `eval_strategy='steps'` / `eval_steps=500` | Run an evaluation pass on the validation set every 500 training steps (instead of only at the end of each epoch), so you can monitor overfitting/underfitting as training progresses. *(Note: this argument used to be called `evaluation_strategy`; it was renamed to `eval_strategy` in newer `transformers` versions — you'll see a `FutureWarning` about this below if the installed version still recognizes the old name.)* |
| `save_steps=1e6` | Save a checkpoint every 1,000,000 steps — effectively "never save an intermediate checkpoint" for this short demo run, since we save the final model manually later instead. |
| `gradient_accumulation_steps=16` | **The most important trick for training large models on limited GPU memory.** Instead of updating weights after every single batch, gradients are accumulated (summed) over 16 consecutive batches before one optimizer step is taken. This *simulates* an effective batch size of `1 × 16 = 16` without ever needing 16 examples' worth of activations in GPU memory at once — a direct trade of extra compute time for reduced memory footprint. |

Larger effective batch sizes (via bigger `per_device_batch_size` and/or more `gradient_accumulation_steps`) generally give more stable gradient estimates and better final quality, at the cost of more GPU memory or more wall-clock time respectively — a very common tuning knob in real projects.

In [ ]:
trainer_args = TrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_strategy='steps',  # <-- RENAMED TO eval_strategy
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


### 7.3 Instantiating the `Trainer`

`Trainer` wires everything together: the model to train, the hyperparameters (`args`), the tokenizer (needed so it can correctly pad/save alongside the model), the `data_collator` (batch-building logic from §6.4), and the two datasets (`train_dataset` for weight updates, `eval_dataset` for periodic validation).

In [ ]:
trainer = Trainer(model=model_pegasus, args=trainer_args,
                  tokenizer=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["test"],
                  eval_dataset=dataset_samsum_pt["validation"])

In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:2618: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=51, training_loss=3.052001864302392, metrics={'train_runtime': 141.6178, 'train_samples_per_second': 5.783, 'train_steps_per_second': 0.36, 'total_flos': 313450454089728.0, 'train_loss': 3.052001864302392, 'epoch': 0.9963369963369964})

### 7.4 Reading the `TrainOutput`

- **`global_step=51`** — total number of optimizer weight updates performed. With `gradient_accumulation_steps=16`, that's 51 × 16 ≈ 816 training examples worth of gradient seen — which lines up almost exactly with `train_dataset` having 819 rows (one epoch).
- **`training_loss=3.05`** — the average cross-entropy loss over training. Lower is better; this number alone is only meaningful *relative to itself* over time (is it decreasing across steps?) — we'll get a more interpretable quality signal from ROUGE in the Evaluation section next.
- **`train_runtime` / `train_samples_per_second` / `train_steps_per_second`** — wall-clock throughput stats, useful for estimating how long a full-scale run would take.

### ⚠️ Important methodology callout: this run trains on the *test* split, not *train*

Look closely at the `Trainer(...)` call above:

```python
train_dataset=dataset_samsum_pt["test"],       # ← should normally be ["train"]
eval_dataset=dataset_samsum_pt["validation"],
```

This is a deliberate shortcut for a **fast tutorial demo**: the `test` split (819 rows) is far smaller than `train` (14,732 rows), so training on it finishes in ~2 minutes instead of much longer. It is **not** correct practice for a real project, for two reasons:

1. **You're throwing away 94% of your labeled data** (`train`) that exists specifically to teach the model.
2. **Data leakage**: later in this notebook we evaluate the fine-tuned model's ROUGE score on `dataset_samsum['test']` — the *same* split it was just trained on. A model scored against data it has already seen during training will look better than it would on genuinely unseen data, undermining the whole point of a held-out test set.

For a real fine-tuning run, swap this to `train_dataset=dataset_samsum_pt["train"]` and expect training to take proportionally longer (and probably want more than 1 epoch — see the Next Steps section at the end of this notebook).

## 8. Model Evaluation with ROUGE

### 8.1 What is ROUGE?

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) is the standard automatic metric for summarization quality. It measures **n-gram overlap** between a generated summary (hypothesis) and one or more human-written reference summaries:

| Metric | What it measures |
|---|---|
| **ROUGE-1** | Overlap of individual words (unigrams) between hypothesis and reference. |
| **ROUGE-2** | Overlap of consecutive word pairs (bigrams) — stricter, rewards matching phrasing, not just vocabulary. |
| **ROUGE-L** | Longest Common Subsequence (LCS) between hypothesis and reference — captures sentence-level structural similarity without requiring exact consecutive matches. |
| **ROUGE-Lsum** | Same LCS idea, but applied summary-line-by-line (splits on newlines first) — more appropriate when a summary has multiple sentences/lines. |

Each is typically reported as an **F1 score** (harmonic mean of precision and recall over the overlapping n-grams):
- **Precision** = (overlapping n-grams) / (n-grams in the *generated* summary) — "of what the model said, how much was relevant?"
- **Recall** = (overlapping n-grams) / (n-grams in the *reference* summary) — "of what the reference said, how much did the model capture?"

ROUGE is cheap to compute and correlates reasonably well with human judgment for extractive-leaning summaries, but it's an imperfect proxy — it can't recognize a *paraphrase* that uses completely different words to express the same meaning, and abstractive models (like ours) are specifically prone to being penalized for legitimate paraphrasing. Treat ROUGE as a useful automatic signal for comparing model versions, not as ground truth for "is this a good summary?" (reading actual generated examples, as we do later, remains essential).

### 8.2 The batch-evaluation helper functions

```python
def generate_batch_sized_chunks(list_of_elements, batch_size):
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]
```
A simple generator that slices a list into fixed-size chunks, e.g. `[1,2,3,4,5,6]` with `batch_size=3` → `[1,2,3]`, `[4,5,6]`. We need this because generating summaries for hundreds of examples one-at-a-time would be slow — batching lets the GPU process several dialogues in parallel per `model.generate()` call.

```python
def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                               column_text="article", column_summary="highlights"):
    ...
    summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                     attention_mask=inputs["attention_mask"].to(device),
                     length_penalty=0.8, num_beams=8, max_length=128)
    ...
```

Key generation settings here, both controlling **beam search decoding**:
- **`num_beams=8`** — instead of greedily picking the single most-likely next token at each step (which can lock in a bad early choice), *beam search* keeps the top-8 most promising partial sequences ("beams") expanded in parallel at every step, and returns the overall best-scoring complete sequence at the end. This generally produces noticeably better-quality text than greedy decoding, at the cost of ~8× more compute per generation step.
- **`length_penalty=0.8`** — during beam search, sequence scores are (by default) a sum of log-probabilities, which naturally favors *shorter* sequences (each extra token multiplies in another <1 probability). The length penalty exponent rebalances this: values **< 1** further discourage long outputs (favor brevity), values **> 1** encourage longer outputs. `0.8` here nudges the model toward concise summaries, appropriate for our short-summary task.

After generating, the function decodes the token IDs back to text, does light cleanup, and streams `(prediction, reference)` pairs into the `metric` object's running total via `metric.add_batch(...)`, finally calling `metric.compute()` to get the aggregated ROUGE scores across the *whole* evaluation set.

In [ ]:
# Evaluation
### lst[1,2,3,4,5,6]-> [1,2,3][4,5,6]
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]



def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                               column_text="article",
                               column_summary="highlights"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024,  truncation=True,
                        padding="max_length", return_tensors="pt")

        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                         attention_mask=inputs["attention_mask"].to(device),
                         length_penalty=0.8, num_beams=8, max_length=128)
        ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''

        # Finally, we decode the generated texts,
        # replace the  token, and add the decoded texts with the references to the metric.
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)
               for s in summaries]

        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]


        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    #  Finally compute and return the ROUGE scores.
    score = metric.compute()
    return score


In [ ]:
!pip install evaluate

...


In [ ]:
import evaluate

rouge_metric = evaluate.load('rouge')
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
#rouge_metric = load_metric('rouge')

In [ ]:
rouge_metric

EvaluationModule(name: "rouge", module_type: "metric", features: [{'predictions': Value(dtype='string', id='sequence'), 'references': Sequence(feature=Value(dtype='string', id='sequence'), length=-1, id=None)}, {'predictions': Value(dtype='string', id='sequence'), 'references': Value(dtype='string', id='sequence')}], usage: """
Calculates average rouge scores for a list of hypotheses and references
Args:
    predictions: list of predictions to score. Each prediction
        should be a string with tokens separated by spaces.
    references: list of reference for each prediction. Each
        reference should be a string with tokens separated by spaces.
    rouge_types: A list of rouge types to calculate.
    use_stemmer: Bool indicating whether Porter stemmer should be used to strip word suffixes.
    use_aggregator: Return aggregates if this is set to True
Returns:
    rouge1: rouge_1 (f1),
    rouge2: rouge_2 (f1),
    rougeL: rouge_l (f1),
    rougeLsum: rouge_lsum (f1)
Examples:

 

In [ ]:
score = calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
)

# Directly use the scores without accessing fmeasure or mid
rouge_dict = {rn: score[rn] for rn in rouge_names}

# Convert the dictionary to a DataFrame for easy visualization
import pandas as pd
pd.DataFrame(rouge_dict, index=[f'pegasus'])

100%|██████████| 5/5 [00:10<00:00,  2.17s/it]


           rouge1  rouge2    rougeL  rougeLsum
pegasus  0.022424     0.0  0.022151    0.02206

### Interpreting Good vs. Bad ROUGE Scores

1. **Scores close to 1**: strong overlap between the generated and reference summaries — desirable. An F1 of 0.7+ across metrics is generally considered good.
2. **Scores between 0.5 and 0.7**: moderate overlap. The summary is probably capturing key points but missing some structure or detail.
3. **Scores below 0.5**: poor match — the model is likely generating irrelevant or incomplete summaries.

### Why is our score (~0.02) so low, given we just trained on the test set?

This looks paradoxical at first — we trained on (a small slice of) the exact split we're now evaluating on, yet the score is very poor. Two compounding reasons explain it, both pointing back to the ⚠️ callout in §7.4:

1. **Severe under-training.** `global_step=51` means only 51 optimizer updates ever happened — nowhere near enough to meaningfully shift half a billion parameters, even on data the model "saw." One epoch over 819 examples with batch size 1 (accumulated to 16) is a smoke test, not a real training run.
2. **Beam-search generation artifacts** (visible in §9 below — look for the stray `Amanda:`/`Hannah:` speaker labels and the literal `<n>` token bleeding into the output) show the model is still largely reproducing raw dialogue lines rather than truly summarizing, which tanks n-gram overlap with the clean reference summaries.

**What a properly-trained run would look like:** train on `dataset_samsum_pt["train"]` (all 14,732 examples) for several epochs, then evaluate on the *held-out* `test` split the model never saw during training. Published SAMSum benchmarks with a properly fine-tuned Pegasus typically reach **ROUGE-1 ≈ 0.47, ROUGE-2 ≈ 0.23, ROUGE-L ≈ 0.39** — night-and-day different from the ~0.02 we got from this 2-minute demo run. The low score here is expected and fine for a first pass through the pipeline; just don't mistake it for "Pegasus is bad at this task."

## 9. Saving & Loading the Model

`save_pretrained(...)` writes everything needed to reload the model later without needing internet access or the original checkpoint name: the architecture config (`config.json`), the learned weights (`pytorch_model.bin` or `model.safetensors`), and the generation defaults (`generation_config.json`). We save the **model** and the **tokenizer** as two separate artifacts (they're loaded independently, but must always be paired together at inference time — mixing a fine-tuned model with an unrelated tokenizer will silently produce garbage, since token IDs only mean what a *specific* tokenizer's vocabulary says they mean).

In [ ]:
## Save model
model_pegasus.save_pretrained("pegasus-samsum-model")

In [ ]:
## Save tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json',
 'tokenizer/special_tokens_map.json',
 'tokenizer/spiece.model',
 'tokenizer/added_tokens.json',
 'tokenizer/tokenizer.json')

In [ ]:
#Load

tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

**Portability note:** `/content/tokenizer` is a Colab-specific absolute path (Colab's working directory is always `/content`). If you run this notebook locally or on a different platform, use a relative path (`"tokenizer"`) or the actual local path instead — hardcoded Colab paths are one of the most common reasons a "working" notebook fails elsewhere. In a production setting you'd typically push these artifacts to a model registry (e.g. the Hugging Face Hub via `push_to_hub()`, or an internal artifact store) rather than relying on a local path at all.

## 10. Inference / Deployment

Now we use the saved model exactly the way an end user or downstream application would: through the `pipeline("summarization", ...)` convenience API introduced in §4.

```python
gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length": 128}
pipe = pipeline("summarization", model="pegasus-samsum-model", tokenizer=tokenizer)
pipe(sample_text, **gen_kwargs)[0]["summary_text"]
```

`gen_kwargs` are passed straight through to `model.generate(...)` — the same beam-search parameters explained in §8.2 (`num_beams=8` for higher-quality search over possible summaries, `length_penalty=0.8` to favor concise output). Keeping these consistent between training-time evaluation and inference-time usage matters: different decoding settings can noticeably change output style and quality even for the *same* trained model.

In [ ]:
gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}



sample_text = dataset_samsum["test"][0]["dialogue"]

reference = dataset_samsum["test"][0]["summary"]

pipe = pipeline("summarization", model="pegasus-samsum-model",tokenizer=tokenizer)

##
print("Dialogue:")
print(sample_text)


print("\nReference Summary:")
print(reference)


print("\nModel Summary:")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Your max_length is set to 128, but your input_length is only 122. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)


Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Amanda: Ask Larry Amanda: He called her last time we were at the park together .<n>Hannah: I'd rather you texted him .<n>Amanda: Just text him .


### Reading this output critically

Two things worth noticing as a learner, not just accepting at face value:

1. **`pipe` silently ran on CPU**, per the warning: *"Hardware accelerator e.g. GPU is available... Model will be on CPU."* The `pipeline(...)` call never received a `device=` argument, so it defaulted to CPU even though `device == "cuda"` was available. Fix: `pipeline("summarization", model=..., tokenizer=..., device=0)` (or `device=device` after mapping `"cuda"`→`0`) to actually use the GPU — easy to miss, and a common silent performance bug.
2. **The `<n>` tokens and leaked speaker labels** (`Amanda:`, `Hannah:`) in the "Model Summary" are symptoms of the under-training discussed in §8. `<n>` is Pegasus's internal newline/sentence-separator token from its CNN/DailyMail pretraining (where summaries were bullet-style, one point per line) — a well fine-tuned model learns to stop emitting it as a raw literal for this dataset's prose-style summaries. A simple post-processing cleanup (`.replace("<n>", " ")`) plus **real training on the full train split for more epochs** (as flagged in §7.4) would fix both issues. Compare this messy output to the clean, correct-register demo output back in §4 (from the *already fully fine-tuned* `pegasus-xsum` checkpoint) — that's the quality bar a properly fine-tuned SAMSum model should reach.

## 11. Key Takeaways, Glossary & Next Steps

### 11.1 The full pipeline, end to end

```
Pretrained checkpoint (pegasus-cnn_dailymail)
        │
        ▼  tokenize dialogue → input_ids/attention_mask, tokenize summary → labels
Preprocessed SAMSum dataset
        │
        ▼  Trainer + TrainingArguments (fine-tuning loop, teacher forcing, cross-entropy loss)
Fine-tuned model checkpoint
        │
        ▼  batch-generate summaries with beam search → score with ROUGE
Evaluation report
        │
        ▼  save_pretrained() model + tokenizer
Deployable artifact
        │
        ▼  pipeline("summarization", ...) at inference time
End-user summaries
```

### 11.2 Glossary (concepts this notebook taught, in one place)

| Term | One-line definition |
|---|---|
| Extractive vs. abstractive summarization | Copy-and-stitch existing text vs. generate new paraphrased text. |
| Transformer / self-attention | Architecture where every token attends to every other token, in parallel, instead of sequential RNN processing. |
| Encoder-decoder (seq2seq) | Encoder compresses the input; decoder generates output tokens autoregressively, attending back to the encoder via cross-attention. |
| Tokenization / sub-word tokens | Splitting text into vocabulary units (often smaller than whole words) that a model can embed as vectors. |
| Transfer learning / fine-tuning | Continuing to train an already-pretrained model on a smaller, task-specific dataset instead of training from scratch. |
| Teacher forcing | Feeding the *ground-truth* previous token to the decoder during training, rather than the model's own (possibly wrong) prediction. |
| `-100` label masking | Sentinel value telling the loss function to ignore padded positions in the target sequence. |
| Beam search / `num_beams` | Decoding strategy that tracks several candidate sequences in parallel instead of greedily committing to one token at a time. |
| `length_penalty` | Exponent that rebalances beam search's natural bias toward shorter sequences. |
| Gradient accumulation | Summing gradients over several small batches before one optimizer step, to simulate a larger effective batch size under memory constraints. |
| ROUGE-1/2/L/Lsum | N-gram / longest-common-subsequence overlap metrics comparing generated vs. reference summaries. |
| LoRA / PEFT | Parameter-efficient fine-tuning: freeze the base model, train small low-rank adapter matrices instead — standard for fine-tuning today's much larger LLMs. |

### 11.3 How this connects to the broader GenAI landscape

Everything in this notebook — encoder-decoder Transformer, tokenization, autoregressive generation, fine-tuning — is the same conceptual machinery underlying today's large generative models (GPT-, Claude-, Gemini-style LLMs). The differences are mostly of **degree**, not kind:

| | This notebook (Pegasus) | Modern general-purpose LLMs |
|---|---|---|
| Parameters | ~570M | Tens to hundreds of billions+ |
| Specialization | One task (summarization), via fine-tuning | Many tasks, via natural-language instructions ("prompting") and instruction-tuning/RLHF |
| Adapting to a new task | Requires gathering labeled data + retraining (as we did here) | Often just requires a good prompt, or a handful of examples ("few-shot"), no retraining needed |
| Where it runs | Small enough to fine-tune and self-host on a single GPU | Usually accessed via API; fine-tuning (when offered) is typically PEFT-based, not full fine-tuning |

**When would you still do what we did here (fine-tune a small specialized model) instead of just prompting a large LLM?** Common real-world reasons: much lower inference cost and latency at scale, ability to self-host for data-privacy/compliance reasons, and better consistency/control for one narrow, well-defined task where you have solid training data — vs. the flexibility, zero-shot capability, and lower upfront engineering cost of prompting a general-purpose LLM.

### 11.4 Suggested exercises to deepen understanding

1. **Fix the methodology issues flagged above**: train on `dataset_samsum_pt["train"]` instead of `["test"]`, run for 3+ epochs, and re-evaluate on the untouched test split. Compare the new ROUGE scores to the ~0.02 baseline here.
2. **Swap the base architecture**: try `facebook/bart-large-cnn`, `google/flan-t5-base`, or `google/long-t5-tglobal-base` (for longer inputs) and compare ROUGE + qualitative output style.
3. **Try parameter-efficient fine-tuning**: install the `peft` library and wrap `model_pegasus` with a LoRA adapter (`get_peft_model`) — compare trainable-parameter count, training time, and final quality vs. the full fine-tune done here.
4. **Fix the silent CPU inference bug** in §10 by passing `device=0` to `pipeline(...)`, and benchmark the speed difference.
5. **Build a small deployment**: wrap the saved `pegasus-samsum-model` + `tokenizer` in a FastAPI or Streamlit app with a text box for pasting a conversation and a button to generate a summary.
6. **Compare against zero-shot prompting**: send a few SAMSum test dialogues to a general-purpose LLM with a one-line instruction ("Summarize this conversation in one sentence") and compare its ROUGE scores and qualitative style against this fine-tuned Pegasus model — a hands-on way to feel the fine-tuning-vs-prompting trade-off discussed above.

### 11.5 Where to read more (background, not required to proceed)

- **"Attention Is All You Need"** (Vaswani et al., 2017) — the original Transformer paper.
- **"PEGASUS: Pre-training with Extracted Gap-sentences for Abstractive Summarization"** (Zhang et al., 2020) — the model used throughout this notebook.
- **"SAMSum Corpus: A Human-annotated Dialogue Dataset for Abstractive Summarization"** (Gliwa et al., 2019) — the dataset used throughout this notebook.
- The Hugging Face `transformers` and `datasets` library documentation — the primary reference for every API call made in this notebook.